# F1 Head-to-Head Prediction Model

Binary Random Forest classifier predicting which driver finishes ahead in a head-to-head matchup.  
Dataset: hybrid era 2014–2024.  
Train / test split: 2014–2021 train, 2022–2024 test (temporal, no leakage).

## 1. Imports & Paths

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

%matplotlib inline

DATA_PATH   = '../data/'
MODELS_PATH = '../models/'
os.makedirs(MODELS_PATH, exist_ok=True)

## 2. Load Data

In [2]:
results      = pd.read_csv(f'{DATA_PATH}results.csv')
races        = pd.read_csv(f'{DATA_PATH}races.csv')
drivers      = pd.read_csv(f'{DATA_PATH}drivers.csv')
constructors = pd.read_csv(f'{DATA_PATH}constructors.csv')
status       = pd.read_csv(f'{DATA_PATH}status.csv')
circuits     = pd.read_csv(f'{DATA_PATH}circuits.csv')

## 3. Filter 2014–2024 & Merge

In [3]:
races_f = races[(races['year'] >= 2014) & (races['year'] <= 2024)].copy()
res = results[results['raceId'].isin(races_f['raceId'])].copy()

# Race context
res = res.merge(races_f[['raceId', 'year', 'circuitId', 'name']], on='raceId', how='left')

# Driver names
drivers = drivers.copy()
drivers['driver_name'] = drivers['forename'] + ' ' + drivers['surname']
res = res.merge(drivers[['driverId', 'driverRef', 'driver_name']], on='driverId', how='left')

# Constructor names (suffix avoids collision with race 'name')
res = res.merge(
    constructors[['constructorId', 'name']], on='constructorId', how='left', suffixes=('', '_constructor')
).rename(columns={'name_constructor': 'constructor_name'})

# Status — DNF = not 'Finished' and not '+N Laps' (lapped drivers still classified as finishers)
res = res.merge(status[['statusId', 'status']], on='statusId', how='left')
res['is_dnf'] = (~res['status'].str.match(r'^(Finished|\+\d+ Lap)')).astype(int)

print(f"Dataset: {len(res)} rows | {res['year'].nunique()} seasons | {res['raceId'].nunique()} races")

Dataset: 4626 rows | 11 seasons | 228 races


## 4. Compute Historical Stats

Career and circuit-specific stats are precomputed from the full 2014–2024 dataset.  
These serve as the feature lookup at inference time.

In [4]:
driver_stats = res.groupby('driverId').agg(
    avg_pos  = ('positionOrder', 'mean'),
    win_rate = ('positionOrder', lambda x: (x == 1).mean())
).reset_index()

constructor_stats = res.groupby('constructorId').agg(
    avg_pts = ('points', 'mean')
).reset_index()

circuit_driver_stats = res.groupby(['driverId', 'circuitId']).agg(
    circuit_avg_pos = ('positionOrder', 'mean')
).reset_index()

print(f"Drivers: {len(driver_stats)} | Constructors: {len(constructor_stats)} | Circuit-driver combos: {len(circuit_driver_stats)}")

Drivers: 59 | Constructors: 20 | Circuit-driver combos: 1268


## 5. Build H2H Pair Dataset

For every race, generate all driver pairs (A, B) where A finished ahead of B.  
Each pair produces two symmetric training samples to keep class balance at 50/50:
- (A vs B): features from A's perspective, target = 1  
- (B vs A): features from B's perspective, target = 0

In [5]:
# Attach precomputed stats to each row for fast pair lookup
df = res.merge(driver_stats[['driverId', 'avg_pos', 'win_rate']], on='driverId', how='left')
df = df.merge(constructor_stats[['constructorId', 'avg_pts']], on='constructorId', how='left')
df = df.merge(circuit_driver_stats[['driverId', 'circuitId', 'circuit_avg_pos']], on=['driverId', 'circuitId'], how='left')
df['circuit_avg_pos'] = df['circuit_avg_pos'].fillna(df['avg_pos'])  # fallback to career avg

In [6]:
FEATURE_COLS = [
    'grid_a', 'grid_b',
    'driver_a_avg_pos', 'driver_b_avg_pos',
    'driver_a_circuit_avg_pos', 'driver_b_circuit_avg_pos',
    'constructor_a_avg_pts', 'constructor_b_avg_pts'
]

pairs = []
for race_id, race in df.groupby('raceId'):
    race = race.sort_values('positionOrder').reset_index(drop=True)
    year = int(race.iloc[0]['year'])
    n = len(race)

    for i in range(n):
        for j in range(i + 1, n):
            a = race.iloc[i]  # finished ahead
            b = race.iloc[j]  # finished behind

            # A beats B
            pairs.append({
                'year': year,
                'grid_a': a['grid'],                      'grid_b': b['grid'],
                'driver_a_avg_pos': a['avg_pos'],         'driver_b_avg_pos': b['avg_pos'],
                'driver_a_circuit_avg_pos': a['circuit_avg_pos'], 'driver_b_circuit_avg_pos': b['circuit_avg_pos'],
                'constructor_a_avg_pts': a['avg_pts'],    'constructor_b_avg_pts': b['avg_pts'],
                'a_wins': 1
            })
            # B beats A (symmetric — keeps dataset balanced at 50/50)
            pairs.append({
                'year': year,
                'grid_a': b['grid'],                      'grid_b': a['grid'],
                'driver_a_avg_pos': b['avg_pos'],         'driver_b_avg_pos': a['avg_pos'],
                'driver_a_circuit_avg_pos': b['circuit_avg_pos'], 'driver_b_circuit_avg_pos': a['circuit_avg_pos'],
                'constructor_a_avg_pts': b['avg_pts'],    'constructor_b_avg_pts': a['avg_pts'],
                'a_wins': 0
            })

pairs_df = pd.DataFrame(pairs).dropna()
print(f"H2H pairs: {len(pairs_df)} | Class balance: {pairs_df['a_wins'].mean():.2f}")

H2H pairs: 89372 | Class balance: 0.50


## 6. Temporal Train / Test Split

Train: 2014–2021 | Test: 2022–2024  
Using time-based split to simulate real prediction scenarios.

In [7]:
train_df = pairs_df[pairs_df['year'] <= 2021]
test_df  = pairs_df[pairs_df['year'] >  2021]

X_train, y_train = train_df[FEATURE_COLS], train_df['a_wins']
X_test,  y_test  = test_df[FEATURE_COLS],  test_df['a_wins']

print(f"Train: {len(X_train)} samples ({train_df['year'].min()}–{train_df['year'].max()})")
print(f"Test:  {len(X_test)} samples ({test_df['year'].min()}–{test_df['year'].max()})")

Train: 63570 samples (2014–2021)
Test:  25802 samples (2022–2024)


## 7. Train RandomForestClassifier

In [8]:
clf = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)
clf.fit(X_train, y_train)
print("Training complete.")

Training complete.


## 8. Evaluation

In [9]:
y_pred = clf.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}\n")
print(classification_report(y_test, y_pred, target_names=['B wins', 'A wins']))

Accuracy: 0.7899

              precision    recall  f1-score   support

      B wins       0.79      0.79      0.79     12901
      A wins       0.79      0.79      0.79     12901

    accuracy                           0.79     25802
   macro avg       0.79      0.79      0.79     25802
weighted avg       0.79      0.79      0.79     25802



## 9. Save Artifacts

In [12]:
joblib.dump(clf, f'{MODELS_PATH}rf_h2h.pkl')

meta = {
    'feature_columns':      FEATURE_COLS,
    'driver_stats':         driver_stats[['driverId', 'avg_pos', 'win_rate']],
    'circuit_driver_stats': circuit_driver_stats,
    'constructor_stats':    constructor_stats[['constructorId', 'avg_pts']],
    'global_avg_pos':       float(driver_stats['avg_pos'].mean()),
    'global_avg_pts':       float(constructor_stats['avg_pts'].mean())
}
joblib.dump(meta, f'{MODELS_PATH}model_meta.pkl')

print(f"Saved: {MODELS_PATH}rf_h2h.pkl")
print(f"Saved: {MODELS_PATH}model_meta.pkl")

Saved: ../models/rf_h2h.pkl
Saved: ../models/model_meta.pkl


In [13]:
# Cleaned race results (includes is_dnf for app stats)
clean_cols = [
    'raceId', 'year', 'circuitId', 'name',
    'driverId', 'driver_name', 'driverRef',
    'constructorId', 'constructor_name',
    'grid', 'positionOrder', 'points', 'is_dnf'
]
res[clean_cols].to_csv(f'{DATA_PATH}f1_cleaned.csv', index=False)

# Driver lookup
drv_in = res['driverId'].unique()
drv_lookup = drivers[drivers['driverId'].isin(drv_in)][['driverId', 'forename', 'surname']].copy()
drv_lookup['full_name'] = drv_lookup['forename'] + ' ' + drv_lookup['surname']
drv_lookup.to_csv(f'{DATA_PATH}drivers_lookup.csv', index=False)

# Constructor lookup
con_in = res['constructorId'].unique()
constructors[constructors['constructorId'].isin(con_in)][['constructorId', 'name']].to_csv(
    f'{DATA_PATH}constructors_lookup.csv', index=False
)

# Circuit lookup
cir_in = res['circuitId'].unique()
circuits[circuits['circuitId'].isin(cir_in)][['circuitId', 'name', 'country']].to_csv(
    f'{DATA_PATH}circuits_lookup.csv', index=False
)

print(f"Saved: f1_cleaned.csv ({len(res)} rows)")
print(f"Saved: drivers_lookup.csv ({len(drv_lookup)} drivers)")

Saved: f1_cleaned.csv (4626 rows)
Saved: drivers_lookup.csv (59 drivers)
